# Libs

In [6]:
import pandas as pd
import numpy as np
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score

# 1. Textual data preprocessing

## Load data

In [2]:
df_dataset = pd.read_csv('steam_games.csv', decimal='.')
df_dataset

,appid,name,genres,description,short_description,release_date,positive_ratings,negative_ratings,average_playtime,price
0,10,Counter-Strike,Action,Play the world's number 1 online action game. ...,Play the world's number 1 online action game. ...,2000-11-01,124534,3339,17612,7.19
1,20,Team Fortress Classic,Action,One of the most popular online action games of...,One of the most popular online action games of...,1999-04-01,3318,633,277,3.99
2,30,Day of Defeat,Action,Enlist in an intense brand of Axis vs. Allied ...,Enlist in an intense brand of Axis vs. Allied ...,2003-05-01,3416,398,187,3.99
3,40,Deathmatch Classic,Action,Enjoy fast-paced multiplayer gaming with Death...,Enjoy fast-paced multiplayer gaming with Death...,2001-06-01,1273,267,258,3.99
4,50,Half-Life: Opposing Force,Action,Return to the Black Mesa Research Facility as ...,Return to the Black Mesa Research Facility as ...,1999-11-01,5250,288,624,3.99
...,...,...,...,...,...,...,...,...,...,...
27068,1065230,Room of Pandora,Adventure;Casual;Indie,"<img src=""https://steamcdn-a.akamaihd.net/stea...",The Room of Pandora is a third-person interact...,2019-04-24,3,0,0,2.09
27069,1065570,Cyber Gun,Action;Adventure;Indie,Have you ever been so lonely that no one but y...,Cyber Gun is a hardcore first-person shooter w...,2019-04-23,8,1,0,1.69
27070,1065650,Super Star Blast,Action;Casual;Indie,<strong>Super Star Blast </strong>is a space b...,Super Star Blast is a space based game with ch...,2019-04-24,0,1,0,3.99
27071,1066700,New Yankee 7: Deer Hunters,Adventure;Casual;Indie,Pursue a snow-white deer through an enchanted ...,Pursue a snow-white deer through an enchanted ...,2019-04-17,2,0,0,5.19


- Como os jogos podem pertencer a mais de um gênero e nesses casos estão separados por ";" converti para listas com o objetivo de facilitar a manipulação

In [3]:
## remove unnecessary columns
df_dataset.drop(columns=['appid', 'release_date', 'positive_ratings', 'negative_ratings', 'average_playtime', 'price'], inplace=True, errors='ignore')
## convert genres column to list
df_dataset['genres'] = df_dataset['genres'].str.split(';')
df_dataset

,name,genres,description,short_description
0,Counter-Strike,[Action],Play the world's number 1 online action game. ...,Play the world's number 1 online action game. ...
1,Team Fortress Classic,[Action],One of the most popular online action games of...,One of the most popular online action games of...
2,Day of Defeat,[Action],Enlist in an intense brand of Axis vs. Allied ...,Enlist in an intense brand of Axis vs. Allied ...
3,Deathmatch Classic,[Action],Enjoy fast-paced multiplayer gaming with Death...,Enjoy fast-paced multiplayer gaming with Death...
4,Half-Life: Opposing Force,[Action],Return to the Black Mesa Research Facility as ...,Return to the Black Mesa Research Facility as ...
...,...,...,...,...
27068,Room of Pandora,"[Adventure, Casual, Indie]","<img src=""https://steamcdn-a.akamaihd.net/stea...",The Room of Pandora is a third-person interact...
27069,Cyber Gun,"[Action, Adventure, Indie]",Have you ever been so lonely that no one but y...,Cyber Gun is a hardcore first-person shooter w...
27070,Super Star Blast,"[Action, Casual, Indie]",<strong>Super Star Blast </strong>is a space b...,Super Star Blast is a space based game with ch...
27071,New Yankee 7: Deer Hunters,"[Adventure, Casual, Indie]",Pursue a snow-white deer through an enchanted ...,Pursue a snow-white deer through an enchanted ...


- O título do jogo carrega palavras-chave que podem ajudar a definir o gênero do jogo, por isso optei por incluir também o título na análise. 
- Percebi que a descrição curta já carrega muitas informações relevantes e assim inialmente optei por utilizar ela com objetivo de reduzir ruído para o TF-IDF.

In [ ]:
## remove stopwords and lemmatize text
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def clean_text(text):
    # remove especial characters and lower case
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    # tokenization, stopwords removal, and lemmatization
    tokens = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words]
    return ' '.join(tokens)

# Creating unified corpus for text features title + description
df_dataset['text_feature'] = df_dataset['name'] + " " + df_dataset['short_description']
df_dataset['clean_text'] = df_dataset['text_feature'].apply(clean_text)
df_dataset

,name,genres,description,short_description,text_feature,clean_text
0,Counter-Strike,[Action],Play the world's number 1 online action game. ...,Play the world's number 1 online action game. ...,Counter-Strike Play the world's number 1 onlin...,counterstrike play world number online action ...
1,Team Fortress Classic,[Action],One of the most popular online action games of...,One of the most popular online action games of...,Team Fortress Classic One of the most popular ...,team fortress classic one popular online actio...
2,Day of Defeat,[Action],Enlist in an intense brand of Axis vs. Allied ...,Enlist in an intense brand of Axis vs. Allied ...,Day of Defeat Enlist in an intense brand of Ax...,day defeat enlist intense brand axis v allied ...
3,Deathmatch Classic,[Action],Enjoy fast-paced multiplayer gaming with Death...,Enjoy fast-paced multiplayer gaming with Death...,Deathmatch Classic Enjoy fast-paced multiplaye...,deathmatch classic enjoy fastpaced multiplayer...
4,Half-Life: Opposing Force,[Action],Return to the Black Mesa Research Facility as ...,Return to the Black Mesa Research Facility as ...,Half-Life: Opposing Force Return to the Black ...,halflife opposing force return black mesa rese...
...,...,...,...,...,...,...
27068,Room of Pandora,"[Adventure, Casual, Indie]","<img src=""https://steamcdn-a.akamaihd.net/stea...",The Room of Pandora is a third-person interact...,Room of Pandora The Room of Pandora is a third...,room pandora room pandora thirdperson interact...
27069,Cyber Gun,"[Action, Adventure, Indie]",Have you ever been so lonely that no one but y...,Cyber Gun is a hardcore first-person shooter w...,Cyber Gun Cyber Gun is a hardcore first-person...,cyber gun cyber gun hardcore firstperson shoot...
27070,Super Star Blast,"[Action, Casual, Indie]",<strong>Super Star Blast </strong>is a space b...,Super Star Blast is a space based game with ch...,Super Star Blast Super Star Blast is a space b...,super star blast super star blast space based ...
27071,New Yankee 7: Deer Hunters,"[Adventure, Casual, Indie]",Pursue a snow-white deer through an enchanted ...,Pursue a snow-white deer through an enchanted ...,New Yankee 7: Deer Hunters Pursue a snow-white...,new yankee deer hunter pursue snowwhite deer e...


# 2. TF-IDF Matrix

In [12]:
tfidf = TfidfVectorizer(encoding='utf-8-sig')
X_tfidf = tfidf.fit_transform(df_dataset['clean_text'])
X_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 527437 stored elements and shape (27073, 41804)>

# 3. PCA Dimmensionally Reduction

In [ ]:
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_tfidf.toarray())
X_pca